# Microring weight bank

Schematic reconstruction after Tait et al., Sci. Rep. 2017
([doi:10.1038/s41598-017-07754-z](https://doi.org/10.1038/s41598-017-07754-z)).

A WDM bus feeds a four-ring bank. The summed drop port goes to a
photodetector. Through is terminated. Per-ring taps `d1`..`d4`
stay unwired on purpose.


In [1]:
from optodraw import Scene

def half_size(component):
    probe = Scene(title="", width=2000, height=2000, platform="on_chip")
    probe.add(component)
    x0, y0, x1, y1 = probe.content_bounds()
    return 0.5 * (x1 - x0), 0.5 * (y1 - y0)


In [2]:
def dangling_optical(scene, connected):
    used = {(id(comp), name) for comp, name in connected}
    leftover = []
    for component in scene.walk():
        ports = getattr(component, "ports", None)
        if not callable(ports):
            continue
        for name, port in ports().items():
            if getattr(port, "kind", "optical") in ("electrical", "fiber"):   # a fibre port is the off-chip side
                continue
            if name.startswith(("e", "h", "gnd", "sig")):
                continue
            if (id(component), name) in used:
                continue
            label = getattr(component, "label", None) or component.__class__.__name__
            leftover.append(f"{label}:{name}")
    return leftover


In [3]:
from optodraw import Scene, VerticalGratingCoupler, MicroringWeightBank, OnChipPhotodetector, Terminator
from optodraw.layout import chain_ports

print("bank half size", half_size(MicroringWeightBank(ring_count=4, with_heaters=True, label="weight bank")))
HOP = 20.0


bank half size (np.float64(100.85139468602475), np.float64(29.544999999999998))


In [4]:
scene = Scene(title="Microring weight bank (after Tait 2017)", width=420, height=180, platform="on_chip")
gc = scene.add(VerticalGratingCoupler(center=(0, 0), angle=180, label="WDM in"))
bank = scene.add(MicroringWeightBank(ring_count=4, with_heaters=True, label="weight bank"))
pd = scene.add(OnChipPhotodetector(label="sum PD"))
term = scene.add(Terminator(label="through"))
chain_ports(
    [(gc, "o1", bank, "o1"), (bank, "drop", pd, "o1"), (bank, "o2", term, "o1")],
    gap=HOP, grid=0,
)
scene.route((gc, "o1"), (bank, "o1"), kind="waveguide", path_type="smooth")
scene.route((bank, "drop"), (pd, "o1"), kind="waveguide", path_type="smooth")
scene.route((bank, "o2"), (term, "o1"), kind="waveguide", path_type="smooth")
leftover = dangling_optical(
    scene,
    [(gc, "o1"), (bank, "o1"), (bank, "drop"), (bank, "o2"), (pd, "o1"), (term, "o1")],
)
print("unconnected", leftover)
scene.fit_frame(margin=16)


unconnected ['weight bank:d1', 'weight bank:d2', 'weight bank:d3', 'weight bank:d4']


In [5]:
print("bounds", scene.content_bounds())
scene.preview(show_3d=False)


bounds (np.float64(-20.7975), np.float64(-6.3000000000000025), np.float64(306.2), np.float64(70.19999999999999))


ScenePreview('Microring weight bank (after Tait 2017)', 900x520, 3d=off)